In [54]:
from pathlib import Path

import numpy as np, torch, h5py
 
slide_id='TCGA-E9-A5FL-01Z-00-DX1'[:24]
h5_dir=Path('../data/h5_features')
attn_dir=Path('../data/outputs/co_attention')
h5_matches=sorted(h5_dir.glob(f'{slide_id}*.h5'))
print('H5 match count:', len(h5_matches))
h5_path=h5_matches[0]




def load_coords(h5_path: Path, dataset_key: str = "coords") -> np.ndarray:
    with h5py.File(h5_path, "r") as handle:
        if dataset_key not in handle:
            raise KeyError(f"Dataset '{dataset_key}' not found in {h5_path}")
        coords = handle[dataset_key][()]
    if coords.ndim != 2 or coords.shape[1] != 2:
        raise ValueError("coords must be shape (N, 2)")
    return coords.astype(np.float32)


coords=load_coords(h5_path)
attn_path=None
 
for pattern in (f'{slide_id}*_co_attention.pt', f'{slide_id}*.npy'):
    matches=sorted(attn_dir.glob(pattern))
    
    if matches:
        attn_path=matches[0]
        
        break

print('Attention path:', attn_path)
 
if attn_path is None: raise SystemExit('No attention file found')

if attn_path.suffix.lower()=='.pt':
    payload=torch.load(attn_path, map_location='cpu')
    
    weights=payload['attention'] if isinstance(payload, dict) else payload
    
    weights=weights.detach().cpu().numpy() if isinstance(weights, torch.Tensor) else np.asarray(weights)
else:
    weights=np.load(attn_path)
weights=weights.squeeze()

weights_1d=weights.mean(axis=(0,1)) if weights.ndim==3 else (weights.mean(axis=0) if weights.ndim==2 else weights)

print('H5:', h5_path)
print('coords length:', coords.shape[0])
print('attention raw shape:', weights.shape)
print('attention 1D length:', weights_1d.shape[0])

H5 match count: 1
Attention path: ..\data\outputs\co_attention\TCGA-E9-A5FL-01Z-00-DX1_co_attention.pt
H5: ..\data\h5_features\TCGA-E9-A5FL-01Z-00-DX1.FB810D6A-303E-45DF-BEF1-D9CE83B4417E.h5
coords length: 31194
attention raw shape: (6, 31194)
attention 1D length: 31194


C:\Users\Administrator\AppData\Local\Temp\ipykernel_7140\231046811.py:41: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  payload=torch.load(attn_path, map_location='cpu')


In [62]:
h5_file = Path('../data/h5_features/TCGA-E9-A5FL-01Z-00-DX1.FB810D6A-303E-45DF-BEF1-D9CE83B4417E.h5')
h5_path = Path('../data/h5_features/TCGA-D8-A3Z5-01Z-00-DX2.6196B9FF-F09E-4D44-873D-E53BFB2BF4E1.h5')
# read h5 file
with h5py.File(h5_path, "r") as handle:
    print(handle.keys())
    
    print(handle['coords'].shape)
    print(handle['features'].shape)

<KeysViewHDF5 ['coords', 'features']>
(29679, 2)
(29679, 1536)


In [50]:
# model = torch.load('../data/outputs/fc_projection/projected/TCGA-5T-A9QA-01Z-00-DX1.B4212117-E0A7-4EF2-B324-8396042ACEC1_projected.pt')
model = torch.load('../data/outputs/fc_projection/projected/TCGA-D8-A3Z5-01Z-00-DX2.6196B9FF-F09E-4D44-873D-E53BFB2BF4E1_projected.pt')
model.shape

C:\Users\Administrator\AppData\Local\Temp\ipykernel_7140\3208579779.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model = torch.load('../data/outputs/fc_projection/pro

FileNotFoundError: [Errno 2] No such file or directory: '../data/outputs/fc_projection/projected/TCGA-D8-A3Z5-01Z-00-DX2.6196B9FF-F09E-4D44-873D-E53BFB2BF4E1_projected.pt'

In [ ]:
import pickle
import sys
import os

from __future__ import annotations

import argparse
import csv
import pickle
from pathlib import Path
import sys
from typing import Any, Dict, List, Optional, Tuple, Union

import h5py
import torch
from torch import nn

from co_attention.train_fc_projection import LinearAutoEncoder

# Đi ngược ra thư mục cha (datamining/) và lấy đường dẫn tuyệt đối
root_path = os.path.abspath(os.path.join('..'))
if root_path not in sys.path:
    sys.path.append(root_path)

from pathlib import Path
import torch
from co_attention.co_attention import GenomicGuidedCoAttention

def _build_projection(
	input_dim: int,
	output_dim: int,
	weights_path: Optional[Path],
	device: torch.device,
) -> nn.Module:
	if weights_path is None:
		layer = nn.Linear(input_dim, output_dim).to(device)
		layer.eval()
		return layer

	state = torch.load(weights_path, map_location=device)
	if any(key.startswith("encoder.") for key in state.keys()):
		model = LinearAutoEncoder(input_dim, output_dim).to(device)
		model.load_state_dict(state)
		model.eval()
		return model.encoder
	layer = nn.Linear(input_dim, output_dim).to(device)
	layer.load_state_dict(state)
	layer.eval()
	return layer



# Reuse slide_id and h5_path from previous cells
# TCGA-D8-A3Z5-01Z-00-DX2.6196B9FF-F09E-4D44-873D-E53BFB2BF4E1
slide_id = 'TCGA-D8-A3Z5-01Z-00-DX2.6196B9FF-F09E-4D44-873D-E53BFB2BF4E1'[:24]
h5_path = Path(h5_path) if 'h5_path' in globals() else None
if h5_path is None or not h5_path.exists():
    raise FileNotFoundError('h5_path not set or missing; run Cell 1 first')
print(h5_path)
# Load patch features from H5
with h5py.File(h5_path, 'r') as handle:
    dataset_name = 'features' if 'features' in handle else list(handle.keys())[0]
    patch_features = handle[dataset_name][()]
patch_features = torch.tensor(patch_features, dtype=torch.float32)

# Load genomic features from combined_genomic_features.pkl
genomic_pkl = Path('../data/csv/combined_genomic_features.pkl')

# Mở file ở dạng đọc nhị phân ('rb') và load bằng pickle
with open(genomic_pkl, 'rb') as f:
    genomic_payload = pickle.load(f)

print((genomic_payload[0].shape))  # Chạy thử để xem kiểu dữ liệu
# genomic_payload = torch.load(genomic_pkl)
if isinstance(genomic_payload, dict):
    submitter_id = slide_id[:12]
    if submitter_id not in genomic_payload:
        raise KeyError(f'Submitter id {submitter_id} not found in genomic features')
    genomic_features = genomic_payload[submitter_id]
else:
    genomic_features = genomic_payload[0]
genomic_features = torch.tensor(genomic_features, dtype=torch.float32) if not isinstance(genomic_features, torch.Tensor) else genomic_features

# Ensure dimensions: Q (6, 512), K/V (N, 512)
model = GenomicGuidedCoAttention(dim=genomic_features.shape[-1])
model.eval()

fc_weights = Path('../data/outputs/fc_projection/fc_autoencoder.pt')

if patch_features.shape[-1] != genomic_features.shape[-1]:
    proj = _build_projection(
        patch_features.shape[-1],
        genomic_features.shape[-1],
        fc_weights,
        device="cpu"
    )
    with torch.no_grad():
        patch_features = proj(patch_features)
with torch.no_grad():
    output, attn = model(genomic_features, patch_features, patch_features)

print('Output shape:', tuple(output.shape))

print('Attention shape:', tuple(attn.shape))

..\data\h5_features\TCGA-D8-A3Z5-01Z-00-DX2.6196B9FF-F09E-4D44-873D-E53BFB2BF4E1.h5
torch.Size([6, 512])
Q shape: torch.Size([1, 6, 512]), K shape: torch.Size([1, 29679, 512]), V shape: torch.Size([1, 29679, 512])
Output shape: (6, 512)
Attention shape: (6, 29679)


C:\Users\Administrator\AppData\Local\Temp\ipykernel_7140\1464273517.py:40: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(weights_path, map_location=device

In [17]:
from __future__ import annotations

import argparse
import sys
from pathlib import Path
from typing import Optional, Tuple
import os
import cv2
import h5py
import matplotlib.pyplot as plt
import numpy as np
import openslide
import torch

root_path = os.path.abspath(os.path.join('..'))
if root_path not in sys.path:
    sys.path.append(root_path)

slide_ids = ["TCGA-AN-A0XO-01Z-00-DX1.204E554E-B1A8-41AD-8E39-62484DF4E3CD", "TCGA-E9-A5FL-01Z-00-DX1.FB810D6A-303E-45DF-BEF1-D9CE83B4417E"]

# import torch

def get_fusion_data(slide_id):
    # Placeholder function - replace with actual implementation
    fusion = torch.load(f"../data/outputs/fusion/{slide_id}_fusion_feature.pt")
    return fusion

def get_h5_data(slide_id):
    file_path = f"../data/h5_features/{slide_id}.h5"
    
    # Mở file h5 ở chế độ đọc ('r')
    with h5py.File(file_path, 'r') as f:
        # Trong file h5 thường có các key (dataset name). 
        # Bạn cần thay 'features' bằng tên key thực tế lưu trong file của bạn.
        # Để kiểm tra các key có trong file, bạn có thể dùng: print(list(f.keys()))
        
        dataset_key = 'features' # Thay đổi key này cho phù hợp
        
        # Đọc dữ liệu thành mảng Numpy
        np_data = f[dataset_key][:]
        
        # Nếu mô hình của bạn yêu cầu PyTorch Tensor, hãy chuyển đổi nó
        h5_data = torch.from_numpy(np_data)
        
    return h5_data

for slide_id in slide_ids:
    # print('Current directory:')
    # print(os.getcwd())
    fusion = get_fusion_data(slide_id)
    h5_data = get_h5_data(slide_id)
    print("Slide ID: ", slide_id)
    print("Fusion features shape: ",fusion.shape)
    print("H5 features shape: ",h5_data.shape)

C:\Users\Administrator\AppData\Local\Temp\ipykernel_19684\1214220629.py:25: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  fusion = torch.load(f"../data/outputs/fusion/{slide

Slide ID:  TCGA-AN-A0XO-01Z-00-DX1.204E554E-B1A8-41AD-8E39-62484DF4E3CD
Fusion features shape:  torch.Size([1, 1024])
H5 features shape:  torch.Size([18980, 1536])
Slide ID:  TCGA-E9-A5FL-01Z-00-DX1.FB810D6A-303E-45DF-BEF1-D9CE83B4417E
Fusion features shape:  torch.Size([1, 1024])
H5 features shape:  torch.Size([31194, 1536])


In [ ]:
from __future__ import annotations

import argparse
import sys
from pathlib import Path
from typing import Optional, Tuple

import cv2
import h5py
import matplotlib.pyplot as plt
import numpy as np
import openslide
import torch

root_path = os.path.abspath(os.path.join('..'))
if root_path not in sys.path:
    sys.path.append(root_path)

from wsi_preprocess.data_loader.batch_pipeline import run_gdc_download


def load_coords(h5_path: Path, dataset_key: str = "coords") -> np.ndarray:
    with h5py.File(h5_path, "r") as handle:
        if dataset_key not in handle:
            raise KeyError(f"Dataset '{dataset_key}' not found in {h5_path}")
        coords = handle[dataset_key][()]
    if coords.ndim != 2 or coords.shape[1] != 2:
        raise ValueError("coords must be shape (N, 2)")
    return coords.astype(np.float32)

def load_attention(attn_path: Path, expected_len: Optional[int] = None) -> np.ndarray:
    if attn_path.suffix.lower() == ".pt":
        payload = torch.load(attn_path, map_location="cpu")
        if isinstance(payload, dict):
            if "attention" not in payload:
                raise KeyError("co_attention payload missing 'attention' key")
            weights = payload["attention"]
        else:
            weights = payload
        if isinstance(weights, torch.Tensor):
            weights = weights.detach().cpu().numpy()
        weights = np.asarray(weights)
    else:
        weights = np.load(attn_path)
    weights = weights.squeeze()
    if weights.ndim == 3:
        weights = weights.mean(axis=(0, 1))
    elif weights.ndim == 2:
        weights = weights.mean(axis=0)
    elif weights.ndim != 1:
        raise ValueError("attention_weights must be a 1D array")

    if expected_len is not None:
        if weights.shape[0] > expected_len:
            weights = weights[:expected_len]
        elif weights.shape[0] < expected_len:
            print(weights.shape)
            raise ValueError(
                "coords length must match attention_weights length"
            )
    return weights.astype(np.float32)

def normalize_weights(weights: np.ndarray) -> np.ndarray:
    min_val = float(weights.min())
    max_val = float(weights.max())
    if max_val - min_val < 1e-8:
        return np.zeros_like(weights)
    return (weights - min_val) / (max_val - min_val)

def get_thumbnail(slide: openslide.OpenSlide, level: int) -> Tuple[np.ndarray, Tuple[int, int]]:
    width, height = slide.level_dimensions[level]
    thumbnail = slide.read_region((0, 0), level, (width, height)).convert("RGB")
    return np.array(thumbnail), (width, height)

def scale_coords(
    coords: np.ndarray,
    slide: openslide.OpenSlide,
    level: int,
) -> np.ndarray:
    downsample = slide.level_downsamples[level]
    return coords / float(downsample)

def build_heatmap_overlay(
    coords: np.ndarray,
    weights: np.ndarray,
    canvas_size: Tuple[int, int],
    patch_size: int,
) -> np.ndarray:
    width, height = canvas_size
    overlay = np.zeros((height, width), dtype=np.uint8)

    for (x, y), weight in zip(coords, weights):
        x0 = int(round(x))
        y0 = int(round(y))
        x1 = min(x0 + patch_size, width)
        y1 = min(y0 + patch_size, height)
        value = int(weight * 255)
        cv2.rectangle(overlay, (x0, y0), (x1, y1), color=value, thickness=-1)

    colored = cv2.applyColorMap(overlay, cv2.COLORMAP_JET)
    return colored


def find_file_by_slideid(search_dir: Path, slide_id: str, patterns: Tuple[str, ...]) -> Path:
    match = find_optional_file_by_slideid(search_dir, slide_id, patterns)
    if match is None:
        raise FileNotFoundError(
            f"No files found for slide_id '{slide_id}' in {search_dir}"
        )
    return match


def find_optional_file_by_slideid(
    search_dir: Path,
    slide_id: str,
    patterns: Tuple[str, ...],
) -> Optional[Path]:
    if not search_dir.exists():
        return None
    matches = []
    for pattern in patterns:
        matches.extend(sorted(search_dir.rglob(pattern.format(slide_id=slide_id))))
    return matches[0] if matches else None


def download_wsi_if_needed(
    manifest: Optional[Path],
    gdc_client: Path,
    output_dir: Path,
    slide_id: str,
    patterns: Tuple[str, ...],
) -> None:
    if manifest is None:
        raise ValueError("--manifest is required to download missing WSI files")
    if not manifest.exists():
        raise FileNotFoundError(f"Manifest not found: {manifest}")
    if find_optional_file_by_slideid(output_dir, slide_id, patterns) is not None:
        return

    run_gdc_download(gdc_client, manifest, output_dir, dry_run=False)


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Visualize co-attention weights as a heatmap overlay on WSI thumbnails."
    )
    parser.add_argument(
        "--slide-id",
        type=str,
        default=None,
        help="Slide ID used to resolve WSI/H5/attention paths.",
    )
    parser.add_argument(
        "--wsi",
        type=Path,
        default=None,
        help="Path to WSI (.svs/.tif).",
    )
    parser.add_argument(
        "--h5",
        type=Path,
        default=None,
        help="Path to H5 file with coords.",
    )
    parser.add_argument(
        "--attention",
        type=Path,
        default=None,
        help="Path to .npy file containing attention weights",
    )
    parser.add_argument(
        "--wsi-dir",
        type=Path,
        default=Path("data/raw_wsi"),
        help="Base directory for WSI files when using --slide-id.",
    )
    parser.add_argument(
        "--h5-dir",
        type=Path,
        default=Path("data/h5_features"),
        help="Base directory for H5 files when using --slide-id.",
    )
    parser.add_argument(
        "--attention-dir",
        type=Path,
        default=Path("data/outputs/co_attention"),
        help="Base directory for attention files when using --slide-id.",
    )
    parser.add_argument(
        "--attention-ext",
        type=str,
        default=".npy",
        help="Extension for attention files when using --slide-id.",
    )
    parser.add_argument(
        "--manifest",
        type=Path,
        default=None,
        help="Optional GDC manifest to download WSI if missing.",
    )
    parser.add_argument(
        "--gdc-client",
        type=Path,
        default=Path("tools/gdc-client.exe"),
        help="Path to gdc-client executable for download.",
    )
    parser.add_argument(
        "--level",
        type=int,
        default=2,
        help="WSI level to render (downsampled thumbnail)",
    )
    parser.add_argument(
        "--patch-size",
        type=int,
        default=256,
        help="Patch size at level 0 (used for rectangle size)",
    )
    parser.add_argument(
        "--alpha",
        type=float,
        default=0.5,
        help="Alpha blending value for heatmap overlay",
    )
    parser.add_argument(
        "--output",
        type=Path,
        default=Path("outputs/co_attention_heatmap.png"),
        help="Output heatmap image path",
    )
    return parser.parse_args()


def main() -> None:
    
    slide_id = "TCGA-E9-A5FL-01Z-00-DX1"
    wsi_patterns = ("{slide_id}*.svs", "{slide_id}*.tif", "{slide_id}*.tiff")
    attention_path = Path("data/outputs/co_attention/TCGA-E9-A5FL-01Z-00-DX1._co_attention.pt")
    wsi_path = find_file_by_slideid(
            "data/raw_wsi",
            slide_id,
            patterns=wsi_patterns,
        )
    h5_path = Path("data/h5_features/TCGA-E9-A5FL-01Z-00-DX1.FB810D6A-303E-45DF-BEF1-D9CE83B4417E.h5")

    coords = load_coords(h5_path)
    print(coords.shape)
    weights = load_attention(attention_path, expected_len=coords.shape[0])
    weights = normalize_weights(weights)

    slide = openslide.OpenSlide(str(wsi_path))
    thumbnail, canvas_size = get_thumbnail(slide, args.level)

    scaled_coords = scale_coords(coords, slide, args.level)
    scaled_patch_size = max(1, int(round(args.patch_size / slide.level_downsamples[args.level])))

    heatmap = build_heatmap_overlay(
        scaled_coords,
        weights,
        canvas_size,
        scaled_patch_size,
    )
    blended = cv2.addWeighted(thumbnail, 1 - args.alpha, heatmap, args.alpha, 0)

    args.output.parent.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(args.output), cv2.cvtColor(blended, cv2.COLOR_RGB2BGR))

    plt.figure(figsize=(10, 8))
    plt.imshow(blended)
    plt.axis("off")
    plt.show()


if __name__ == "__main__":
    main()

usage: ipykernel_launcher.py [-h] [--slide-id SLIDE_ID] [--wsi WSI] [--h5 H5]
                             [--attention ATTENTION] [--wsi-dir WSI_DIR]
                             [--h5-dir H5_DIR] [--attention-dir ATTENTION_DIR]
                             [--attention-ext ATTENTION_EXT]
                             [--manifest MANIFEST] [--gdc-client GDC_CLIENT]
                             [--level LEVEL] [--patch-size PATCH_SIZE]
                             [--alpha ALPHA] [--output OUTPUT]
ipykernel_launcher.py: error: unrecognized arguments: --f=c:\Users\Administrator\AppData\Roaming\jupyter\runtime\kernel-v3f88dd20a965578ac9445e6068de8ac10ebe8a167.json


SystemExit: 2